# 03 — Build the final gene tables and pathway diagram

This notebook combines Liu's original gene information with the current database records gathered in Notebook 2. It organizes each gene by its likely place and role in secretion, records how strong the supporting evidence is, and produces the final tables, review files, and pathway diagrams. Unclear matches and descriptions are left visible instead of being guessed.

Load the shared rules and prepare the folders where the final tables and quality checks will be written.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import yaml

sys.path.insert(0, os.path.abspath(".."))
from atlas import crosswalk
from atlas.schema import (
    COLUMN_ORDER, DATA_DICTIONARY, REQUIRED_FIELDS, SUBSYSTEM_ORDER, EvidenceSource,
    GlycosylationRole, MappingStatus, RecordOrigin, RecordType,
)
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../data/processed/qa").mkdir(parents=True, exist_ok=True)
with open("../config/sources.yaml", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

### Keep only clear database matches

Load the crosswalk from Notebook 02 and keep annotations only when an AO090 ID has one clear UniProt candidate. Conflicting candidates remain available for review.

In [2]:
resolved = pd.read_parquet("../data/interim/seed_resolved.parquet")
uniprot_locus = pd.read_parquet("../data/interim/uniprot_locus_crosswalk.parquet")
pathway_members = pd.read_parquet("../data/interim/kegg_pathways.parquet")
kegg_crosswalk = pd.read_parquet("../data/interim/crosswalk.parquet")
assert len(resolved) == 369

candidate_counts = (
    uniprot_locus.groupby("ao_locus_tag")["uniprot_accession"]
    .nunique()
    .rename("annotation_candidate_count")
)
single = (
    uniprot_locus.sort_values(["ao_locus_tag", "uniprot_accession"])
    .drop_duplicates("ao_locus_tag")
    .merge(candidate_counts, on="ao_locus_tag", how="left")
)
# Keep annotations only when a gene has one clear UniProt match. The full
# crosswalk keeps every candidate for review in gene_id_mapping.csv.
annotation_fields = ["kegg_gene_id", "uniprot_accession", "ncbi_gene_id",
                     "gene_name", "function", "compartment_raw"]
single.loc[single.annotation_candidate_count > 1, annotation_fields] = None
single = single.rename(columns={c: f"annotation_{c}" for c in annotation_fields})
candidate_summary = (
    candidate_counts.value_counts().sort_index()
    .rename_axis("UniProt matches for one AO090 ID")
    .reset_index(name="AO090 IDs")
)
print("Most AO090 IDs have one clear UniProt match. IDs with multiple matches are kept for review.")
display(candidate_summary)

Most AO090 IDs have one clear UniProt match. IDs with multiple matches are kept for review.


,UniProt matches for one AO090 ID,AO090 IDs
0,1,12059
1,2,5


### Add current details to Liu's gene list

Join the current database records to Liu's 369 genes, standardize pathway labels, and assign the most specific supported subsystem and cell location. Each assignment keeps its evidence source and confidence.

In [3]:
SUBSYSTEM_MAP = {
    "TC": "tc", "Dolichol pathway": "dolichol_pathway",
    "Erglycosylation": "er_glycosylation", "Folding": "folding",
    "GPI biosynthesis": "gpi_biosynthesis", "ERAD": "erad",
    "COPII": "copii", "COPI": "copi",
    "Golgi processing": "golgi_processing", "LDSV": "ldsv",
    "HDSV": "hdsv", "CPY pathway": "cpy_pathway",
    "ALPpathway": "alp_pathway", "SNARE": "snare",
    "Septin": "septin",
    "beta-1,6 glucan biosynthesis": "beta_1_6_glucan_biosynthesis",
    "Translation": "translation",
    "putative mitochondria protein": "putative_mitochondria_protein",
    "mitochondrial m‐AAA protease": "mitochondrial_m_aaa_protease",
}
machinery = resolved.merge(single, on="ao_locus_tag", how="left", validate="many_to_one")
assert len(machinery) == 369 and machinery.record_id.is_unique
machinery["gene_name"] = machinery["annotation_gene_name"]
machinery["function"] = machinery["Description"].fillna(machinery["annotation_function"])
machinery["uniprot_accession"] = machinery["annotation_uniprot_accession"]
machinery["kegg_gene_id"] = machinery["annotation_kegg_gene_id"]
machinery["ncbi_gene_id"] = machinery["annotation_ncbi_gene_id"]
ko_lookup = (
    kegg_crosswalk.dropna(subset=["kegg_ko"])
    .drop_duplicates("ao_locus_tag")
    .set_index("ao_locus_tag")["kegg_ko"]
)
machinery["kegg_ko"] = machinery.ao_locus_tag.map(ko_lookup)
machinery["yeast_ortholog"] = machinery["S. cerevisiae ortholog"]
# Preserve Liu's four evidence columns as one searchable text field.
source_columns = ["1st SOURCE", "2nd SOURCE", "3rd SOURCE", "4th SOURCE"]
machinery["liu_source_raw"] = machinery[source_columns].apply(lambda row: "|".join("" if pd.isna(value) else str(value) for value in row), axis=1)
machinery["liu_table_row"] = [f"S1:{row}" for row in range(3, 3 + len(machinery))]
machinery["subsystem"] = machinery["Subsystems or function"].map(SUBSYSTEM_MAP)
machinery["pathway_order"] = machinery["subsystem"].map(SUBSYSTEM_ORDER)
before_subsystem_coverage = int(machinery["subsystem"].notna().sum())
feizi = pd.read_excel("../data/raw/liu2014/feizi2013_table_s1.xlsx", sheet_name="Sheet1", header=1)
FEIZI_MAP = {
    "Translocation": "tc", "Dolichol": "dolichol_pathway",
    "ERglycosylation": "er_glycosylation", "protein folding": "folding",
    "GPI biosynthesis": "gpi_biosynthesis", "ERADL": "erad",
    "ERADM": "erad", "ERADC": "erad", "COPII": "copii",
    "COPI": "copi", "Golgi processing": "golgi_processing",
    "LDSV": "ldsv", "HDSV": "hdsv", "CPY pathway": "cpy_pathway",
    "ALP pathway": "alp_pathway", "SNARE": "snare",
}
# The Feizi table connects yeast genes to secretion stages. Use it only when
# a Liu gene has one clear yeast counterpart.
scaffold = feizi.rename(columns={"Standard Name": "yeast_ortholog"})[["yeast_ortholog", "SUBSYSTEM"]]
scaffold["subsystem"] = scaffold["SUBSYSTEM"].map(FEIZI_MAP)
scaffold = scaffold.dropna(subset=["subsystem"])
machinery = crosswalk.assign_subsystems_from_evidence(machinery, scaffold)
CURATED_SUBSYSTEMS = {
    "AO090701000141": {"subsystem": "er_glycosylation", "source": "liu2014_description", "confidence": "high", "rationale": "Liu's description, the UniProt annotation, and KEGG KO evidence identify glucosidase I, which trims the first glucose from N-glycans in the ER; folding and ER-processing memberships are secondary context."},
    "AO090003001225": {"subsystem": "er_glycosylation", "source": "liu2014_description", "confidence": "high", "rationale": "Liu's description, the UniProt annotation, and KEGG KO K01230 identify a class I alpha-mannosidase that performs ER N-glycan mannose trimming; ER quality control is a secondary role."},
    "AO090020000468": {"subsystem": "er_glycosylation", "source": "liu2014_description", "confidence": "high", "rationale": "Liu's description, the UniProt annotation, and KEGG KO K12670 identify WBP1 as the ER oligosaccharyltransferase beta subunit, making N-glycan transfer its most specific function."},
    "AO090009000178": {"subsystem": "erad", "source": "aspergillus_homolog", "confidence": "medium", "rationale": "Aspergillus homolog evidence, the UniProt annotation, yeast MNL2 orthology, and KEGG KO K01230 identify a class I alpha-mannosidase whose ER-localized ortholog functions in ERAD quality control."},
    "AO090003000257": {"subsystem": "folding", "source": "curated_liu_composite", "confidence": "high", "rationale": "KAR2/BiP is the primary ER Hsp70 folding chaperone; its ERAD contribution is secondary quality-control context.", "override_verified_liu": True},
}
machinery = crosswalk.apply_curated_subsystem_resolutions(machinery, CURATED_SUBSYSTEMS)
machinery = crosswalk.assign_missing_subsystems(
    machinery, pathway_members, cfg["kegg_pathway_to_subsystem"]
)
source_confidence = {"liu2014": "high", "liu2014_description": "medium", "yeast_scaffold": "medium", "current_annotation": "medium", "aspergillus_homolog": "medium", "kegg_pathway": "low", "unassigned": "low"}
machinery["subsystem_confidence"] = machinery["subsystem_confidence"].fillna(machinery.subsystem_source.map(source_confidence))
default_rationale = machinery.subsystem_source.map({"liu2014": "Verified Liu 2014 Table S1 subsystem.", "liu2014_description": "Specific function stated in Liu 2014 description.", "yeast_scaffold": "Unique match in the original yeast secretory-system scaffold.", "current_annotation": "Specific current database function annotation.", "kegg_pathway": "Single mapped KEGG pathway; broad, low-confidence placement.", "unassigned": "Available evidence does not distinguish a primary subsystem."})
machinery["subsystem_rationale"] = machinery["subsystem_rationale"].fillna(default_rationale)
after_subsystem_coverage = int(machinery["subsystem"].notna().sum())
source_labels = {
    "liu2014": "Pathway stage stated directly by Liu",
    "liu2014_description": "Assigned from Liu's gene description",
    "yeast_scaffold": "Assigned through the matching yeast gene",
    "kegg_pathway": "Assigned from a KEGG pathway (lower confidence)",
    "current_annotation": "Assigned from a current database description",
    "curated_liu_composite": "Manually reviewed using several Liu clues",
    "aspergillus_homolog": "Assigned from a related Aspergillus gene",
    "unassigned": "Not enough evidence for a pathway stage",
}
source_counts = machinery["subsystem_source"].value_counts(dropna=False)
subsystem_summary = pd.DataFrame({
    "How the pathway stage was assigned": [source_labels[source] for source in source_counts.index],
    "Genes": source_counts.values,
})
print(f"Liu directly placed {before_subsystem_coverage} of 369 genes in a secretion stage.")
print(f"Using the documented supporting sources adds {after_subsystem_coverage - before_subsystem_coverage} placements, for {after_subsystem_coverage} of 369 total. The remaining {len(machinery) - after_subsystem_coverage} genes stay unassigned.")
display(subsystem_summary)
machinery = crosswalk.assign_controlled_compartments(machinery)
machinery["citation"] = "10.1186/1752-0509-8-73"
machinery["record_type"] = RecordType.MACHINERY.value
machinery["record_origin"] = RecordOrigin.LIU2014.value

Liu directly placed 109 of 369 genes in a secretion stage.
Using the documented supporting sources adds 138 placements, for 247 of 369 total. The remaining 122 genes stay unassigned.


,How the pathway stage was assigned,Genes
0,Not enough evidence for a pathway stage,122
1,Pathway stage stated directly by Liu,108
2,Assigned from Liu's gene description,77
3,Assigned through the matching yeast gene,44
4,Assigned from a KEGG pathway (lower confidence),13
5,Assigned from a current database description,3
6,Manually reviewed using several Liu clues,1
7,Assigned from a related Aspergillus gene,1


#### Record evidence, uncertain matches, and sugar-modification roles

Translate Liu's source notes into consistent evidence labels, flag unclear ID matches, and assign conservative glycosylation roles. Expression changes do not increase confidence in a gene's function.

In [4]:
# A gene changing expression tells us that it responded, not what it does.
# Keep functional evidence separate from the expression result.
source_text = machinery[["1st SOURCE", "2nd SOURCE", "3rd SOURCE", "4th SOURCE"]].fillna("").agg(" | ".join, axis=1)
machinery["evidence_source"] = EvidenceSource.UNKNOWN.value
machinery.loc[source_text.str.contains("inparanoid|psi-blast|bhits", case=False), "evidence_source"] = EvidenceSource.YEAST_INFERENCE.value
machinery.loc[source_text.str.contains("Oliveira", case=False), "evidence_source"] = EvidenceSource.ASPERGILLUS_HOMOLOG.value
machinery.loc[source_text.str.contains("wang et al 2010-AO", case=False), "evidence_source"] = EvidenceSource.AO_TRANSCRIPTOMIC.value
machinery.loc[source_text.str.contains("Kuratsu et al 2007-AO SNARE", case=False), "evidence_source"] = EvidenceSource.AO_EXPERIMENTAL.value

multi = machinery.annotation_candidate_count.gt(1)
machinery.loc[multi, "mapping_status"] = MappingStatus.AMBIGUOUS.value
mapping_method_by_status = {
    "exact": "direct_locus_tag", "cross_reference": "cross_reference",
    "sequence": "sequence", "orthology": "orthology",
    "ambiguous": "ambiguous", "split": "split",
    "merged": "merged", "unresolved": "unresolved",
}
machinery["mapping_method"] = machinery.mapping_status.map(mapping_method_by_status)
machinery["manual_review_required"] = machinery["manual_review_required"].fillna(False) | multi
machinery["manual_review_required"] |= machinery.mapping_status.isin(["unresolved", "ambiguous", "split", "merged"])

# Conservative role rules always retain their provenance and confidence.
machinery = crosswalk.assign_glycosylation_roles(machinery, cfg["glycosylation_ko"])
trimming_tags = ["AO090701000141", "AO090003001225", "AO090009000178"]
machinery.loc[machinery.liu_ao_locus_tag.isin(trimming_tags), ["glycosylation_role", "glycosylation_role_source", "glycosylation_role_confidence"]] = ["n_glycan_trimming", "curated_multi_source_evidence", "high"]
machinery.loc[machinery.liu_ao_locus_tag.eq("AO090020000468"), ["glycosylation_role", "glycosylation_role_source", "glycosylation_role_confidence"]] = ["n_glycan_transfer", "curated_multi_source_evidence", "high"]
glycosylation_labels = {
    "none": "No glycosylation role assigned in this atlas",
    "gpi_anchor": "GPI-anchor production",
    "n_glycan_assembly": "N-glycan assembly",
    "golgi_mannosylation": "Mannose addition in the Golgi",
    "n_glycan_transfer": "Transfer of an N-glycan to a protein",
    "n_glycan_trimming": "Trimming of an N-glycan",
}
glycosylation_counts = machinery.glycosylation_role.value_counts()
glycosylation_summary = pd.DataFrame({
    "Glycosylation role": [glycosylation_labels[role] for role in glycosylation_counts.index],
    "Genes": glycosylation_counts.values,
})
print("A missing role means this project did not assign one; it does not prove the gene has no glycosylation function.")
display(glycosylation_summary)

A missing role means this project did not assign one; it does not prove the gene has no glycosylation function.


,Glycosylation role,Genes
0,No glycosylation role assigned in this atlas,307
1,GPI-anchor production,20
2,N-glycan assembly,14
3,Mannose addition in the Golgi,12
4,Transfer of an N-glycan to a protein,9
5,Trimming of an N-glycan,7


### Note what is outside this atlas

Liu's predicted secretome is not processed here because this atlas is limited to the 369 secretion-machinery genes.

In [5]:
# Liu's predicted secretome remains available in supplementary Table S3 and is out of scope for this atlas.

### Check and save the final tables

Build the final atlas, shortlist, mapping table, glycosylation table, review queue, and data dictionary. Before saving them, confirm that all 369 genes remain unique and that the 51-gene response result is unchanged.

In [6]:
# Four genes have a well-supported second role in addition to their primary
# pathway placement. Keep that extra role on the same one-gene row.
secondary = {
    "AO090701000141": "folding",
    "AO090003001225": "folding",
    "AO090020000468": "folding",
    "AO090009000178": "er_glycosylation",
}
secondary_rationale = "Secondary role supported by the reviewed functional context retained during primary subsystem curation."
machinery["secondary_subsystem"] = machinery.liu_ao_locus_tag.map(secondary)
machinery["secondary_subsystem_rationale"] = machinery.secondary_subsystem.notna().map({True: secondary_rationale, False: None})
machinery_final = machinery.reindex(columns=COLUMN_ORDER)
assert len(machinery_final) == 369
assert machinery_final.record_id.is_unique
assert machinery_final.secondary_subsystem.notna().sum() == 4
for field in REQUIRED_FIELDS:
    assert machinery_final[field].notna().all(), f"machinery: null {field}"
assert int(machinery_final.sig_all_three.sum()) == 51
assert machinery_final.loc[machinery_final.sig_all_three, "direction_all_three"].value_counts().to_dict() == {"up": 48, "down": 3}

# Write the main atlas and the smaller tables intended for direct use.
machinery_final.to_csv("../data/processed/secretion_machinery_genes.csv", index=False)
uniprot_locus.to_csv("../data/processed/gene_id_mapping.csv", index=False)
shortlist_columns = [
    "ao_locus_tag", "gene_name", "yeast_ortholog", "direction_all_three",
    "function", "subsystem", "subsystem_source", "compartment",
    "glycosylation_role", "glycosylation_role_source",
    "glycosylation_role_confidence", "evidence_source", "mapping_status",
    "manual_review_required",
]
shortlist = (
    machinery_final.loc[machinery_final.sig_all_three, shortlist_columns]
    .rename(columns={"direction_all_three": "direction"})
)
shortlist.to_csv("../data/processed/high_secretion_responsive_genes.csv", index=False)
confidence = source_confidence
subsystem_audit = machinery[["record_id", "liu_ao_locus_tag", "yeast_ortholog", "Subsystems or function", "subsystem", "subsystem_source", "subsystem_evidence", "subsystem_candidates", "pathway_order", "manual_review_required"]].copy()
subsystem_audit["assignment_confidence"] = subsystem_audit.subsystem_source.map(confidence)
subsystem_audit = subsystem_audit.rename(columns={"liu_ao_locus_tag": "locus_tag", "Subsystems or function": "liu_subsystem", "subsystem": "final_subsystem", "subsystem_source": "assignment_source", "subsystem_evidence": "evidence_summary", "subsystem_candidates": "conflicting_candidates"})
glyco = machinery_final.loc[machinery_final.glycosylation_role.ne("none"), ["ao_locus_tag", "gene_name", "yeast_ortholog", "function", "glycosylation_role", "glycosylation_role_source", "glycosylation_role_confidence", "subsystem", "compartment"]]
glyco.to_csv("../data/processed/glycosylation_genes.csv", index=False)
# Build one review queue for unclear IDs, annotations, and rejected pathway
# candidates so a person can inspect them without searching the full atlas.
review = machinery_final[machinery_final.manual_review_required].copy()
review = review[["record_id", "record_type", "liu_ao_locus_tag", "gene_name", "yeast_ortholog", "function", "mapping_status", "mapping_method", "subsystem", "subsystem_source", "manual_review_required"]].rename(columns={"liu_ao_locus_tag": "source_locus_tag", "function": "plain_english_function"})
candidate_lookup = uniprot_locus.groupby("ao_locus_tag")["uniprot_accession"].agg(lambda x: "; ".join(sorted(set(x.dropna())))).to_dict()
review["candidate_identifiers"] = review.source_locus_tag.map(candidate_lookup).fillna("none found")
review["review_row_type"] = "review_needed"
review["issue_type"] = "conflicting subsystem evidence"
review.loc[review.mapping_status.eq("unresolved"), "issue_type"] = "unresolved current identifier"
review.loc[review.mapping_status.eq("ambiguous"), "issue_type"] = "ambiguous current identifier"
review["recommended_action"] = "Review the named evidence; retain unassigned until one interpretation is supported."
rejected = subsystem_audit[subsystem_audit.conflicting_candidates.fillna("").str.strip().ne("")].copy()
rejected["review_row_type"] = "rejected_candidate"
rejected["source_locus_tag"] = rejected.locus_tag
rejected["issue_type"] = "rejected subsystem candidate"
rejected["recommended_action"] = "Retain the rejected candidate for audit; do not promote it without additional support."
review = pd.concat([review, rejected], ignore_index=True, sort=False)
review.to_csv("../data/processed/qa/genes_needing_review.csv", index=False)
# Save a small comparison sample that shows original Liu values beside the
# fields derived by this project.
inspection_ids = ["AO090003000257", "AO090120000461", "AO090001000698", "AO090103000069", "AO090001000733", "AO090012000213"]
extra_ids = machinery_final.loc[~machinery_final.liu_ao_locus_tag.isin(inspection_ids), "liu_ao_locus_tag"].head(14).tolist()
sample = machinery[machinery.liu_ao_locus_tag.isin(inspection_ids + extra_ids)][["liu_ao_locus_tag", "S. cerevisiae ortholog", "Subsystems or function", "Description", "subsystem", "subsystem_source", "subsystem_confidence", "subsystem_rationale", "function", "sig_all_three", "direction_all_three"]].copy()
sample["_order"] = sample.liu_ao_locus_tag.map({tag: i for i, tag in enumerate(inspection_ids + extra_ids)})
sample = sample.sort_values("_order").drop(columns="_order").rename(columns={"liu_ao_locus_tag": "Liu ID", "S. cerevisiae ortholog": "Liu yeast ortholog", "Subsystems or function": "Liu subsystem cell", "Description": "Liu description", "subsystem": "derived subsystem", "subsystem_source": "derived subsystem source", "subsystem_confidence": "derived confidence", "subsystem_rationale": "derived rationale", "function": "derived function"})
assert len(sample) == 20
sample.to_csv("../data/processed/qa/liu_vs_atlas_sample.csv", index=False)
# Write the complete field guide, then refresh the short version embedded in
# the README from the same definitions so the two cannot drift apart.
column_descriptions = pd.DataFrame(
    [{"field": field, **details} for field, details in DATA_DICTIONARY.items()]
)
column_descriptions.to_csv("../data/processed/column_descriptions.csv", index=False)
readme_fields = ["evidence_source", "subsystem_source", "mapping_status", "sig_all_three", "glycosylation_role", "manual_review_required"]
readme_dictionary = column_descriptions.set_index("field").loc[readme_fields].reset_index()
dictionary_lines = ["| Field | Definition | Allowed values |", "|---|---|---|"]
for row in readme_dictionary[["field", "definition", "allowed_values"]].itertuples(index=False):
    cells = [str(value).replace("|", "&#124;").replace("\n", " " ) for value in row]
    dictionary_lines.append("| " + " | ".join(cells) + " |")
readme_path = Path("../README.md")
readme = readme_path.read_text(encoding="utf-8")
start, end = "<!-- DATA_DICTIONARY_START -->", "<!-- DATA_DICTIONARY_END -->"
before, remainder = readme.split(start, 1)
_, after = remainder.split(end, 1)
_ = readme_path.write_text(before + start + "\n" + "\n".join(dictionary_lines) + "\n" + end + after, encoding="utf-8")

#### Summarize coverage and unresolved gaps

Record coverage, mapping, annotation, and review counts in one audit file so gaps in the release remain visible.

In [7]:
audit = {
    "machinery_rows": len(machinery_final),
    "machinery_mapping_status": machinery_final.mapping_status.value_counts().to_dict(),
    "machinery_with_uniprot": int(machinery_final.uniprot_accession.notna().sum()),
    "machinery_with_ncbi": int(machinery_final.ncbi_gene_id.notna().sum()),
    "machinery_with_kegg_gene": int(machinery_final.kegg_gene_id.notna().sum()),
    "machinery_with_ko": int(machinery_final.kegg_ko.notna().sum()),
    "kegg_pathway_membership_rows": len(pathway_members),
    "machinery_with_subsystem": int(machinery_final.subsystem.notna().sum()),
    "machinery_subsystem_source": machinery_final.subsystem_source.value_counts().to_dict(),
    "subsystem_conflict_rows": int((
        machinery_final.subsystem_source.eq("unassigned")
        & machinery_final.manual_review_required
        & machinery_final.mapping_status.eq("exact")
    ).sum()),
    "machinery_with_glycosylation_role": int(machinery_final.glycosylation_role.ne("none").sum()),
    "transcriptomic_shortlist_rows": len(shortlist),
    "review_queue_rows": len(review),
    "important_limitations": [
        "KEGG does not provide a functional-group link for every gene; missing values remain blank",
        "A broad KEGG pathway is lower-confidence evidence for a gene's secretion stage and should be reviewed",
        "No assigned glycosylation role means not identified in this pass, not experimentally proven absent",
        "An expression response does not make a gene's functional description more certain",
        "When UniProt provides several candidates, all are withheld from the final gene row and flagged for review",
    ],
}
with open("../data/processed/qa/build_audit.json", "w", encoding="utf-8") as handle:
    json.dump(audit, handle, indent=2)
audit_summary = pd.DataFrame([
    ("Genes in the final atlas", audit["machinery_rows"]),
    ("Genes with a pathway stage", audit["machinery_with_subsystem"]),
    ("Genes with a KEGG functional group", audit["machinery_with_ko"]),
    ("Genes with an assigned glycosylation role", audit["machinery_with_glycosylation_role"]),
    ("Genes responsive in all three strains", audit["transcriptomic_shortlist_rows"]),
    ("Rows in the manual-review queue", audit["review_queue_rows"]),
], columns=["Build check", "Count"])
print("Final build summary:")
display(audit_summary)
print("Important limitations:")
for limitation in audit["important_limitations"]:
    print(f"- {limitation}")

Final build summary:


,Build check,Count
0,Genes in the final atlas,369
1,Genes with a pathway stage,247
2,Genes with a KEGG functional group,330
3,Genes with an assigned glycosylation role,62
4,Genes responsive in all three strains,51
5,Rows in the manual-review queue,14


Important limitations:
- KEGG does not provide a functional-group link for every gene; missing values remain blank
- A broad KEGG pathway is lower-confidence evidence for a gene's secretion stage and should be reviewed
- No assigned glycosylation role means not identified in this pass, not experimentally proven absent
- An expression response does not make a gene's functional description more certain
- When UniProt provides several candidates, all are withheld from the final gene row and flagged for review


### Build the pathway diagram and coverage summary

Count genes and responsive genes by subsystem, then render the SVG and PNG pathway diagrams directly from the final atlas. Also write a short coverage summary explaining assigned and unassigned genes.

In [8]:
# Count total and responsive genes for every pathway stage shown in the figure.
counts = (
    machinery_final.groupby("subsystem", dropna=False)
    .agg(
        genes=("record_id", "size"),
        responsive=("sig_all_three", "sum"),
        up=("direction_all_three", lambda values: (values == "up").sum()),
        down=("direction_all_three", lambda values: (values == "down").sum()),
    )
    .reset_index()
)
count_lookup = {
    row.subsystem: (int(row.genes), int(row.responsive), int(row.up), int(row.down))
    for row in counts.itertuples()
}
unassigned_count = int(machinery_final.subsystem.isna().sum())
unassigned_rows = machinery_final[machinery_final.subsystem.isna()]
unassigned_stats = (unassigned_count, int(unassigned_rows.sig_all_three.sum()), int((unassigned_rows.direction_all_three == "up").sum()), int((unassigned_rows.direction_all_three == "down").sum()))
main_flow = ["tc", "dolichol_pathway", "er_glycosylation", "folding", "gpi_biosynthesis", "erad", "copii", "copi", "golgi_processing", "ldsv", "hdsv", "snare"]
branches = ["cpy_pathway", "alp_pathway", "septin", "beta_1_6_glucan_biosynthesis"]
other = ["translation", "putative_mitochondria_protein", "mitochondrial_m_aaa_protease", "unassigned / uncertain"]
labels = main_flow + branches + other
label_map = {
    "tc": "Translocation (tc)", "dolichol_pathway": "Dolichol-linked glycan assembly",
    "er_glycosylation": "ER N-glycosylation", "folding": "Protein folding",
    "gpi_biosynthesis": "GPI-anchor biosynthesis (gpi)",
    "erad": "ER-associated degradation (erad)",
    "copii": "ER-to-Golgi vesicles (COPII)", "copi": "Golgi-to-ER vesicles (COPI)",
    "golgi_processing": "Golgi glycan processing",
    "ldsv": "Low-density secretory vesicles (ldsv)",
    "hdsv": "High-density secretory vesicles (hdsv)",
    "cpy_pathway": "Vacuolar sorting, CPY route",
    "alp_pathway": "Vacuolar sorting, ALP route",
    "snare": "SNARE vesicle fusion", "septin": "Septin organization",
    "beta_1_6_glucan_biosynthesis": "Beta-1,6-glucan biosynthesis",
    "translation": "Translation",
    "putative_mitochondria_protein": "Putative mitochondrial protein",
    "mitochondrial_m_aaa_protease": "Mitochondrial m-AAA protease",
    "unassigned / uncertain": "Unassigned / uncertain",
}
from datetime import date
import textwrap
build_date = date.today().isoformat()
subtitle_1 = 'Each box is a stage of the secretion pathway. "Responsive" = changed significantly in all three'
subtitle_2 = 'alpha-amylase-overproducing strains (Liu 2014, adj. p < 0.05).'
box_w, box_h, gap = 205, 126, 28
width, height = 2870, 760
positions = {label: (45 + i * (box_w + gap), 150) for i, label in enumerate(main_flow)}
positions.update({"cpy_pathway": (1909, 345), "alp_pathway": (2142, 345), "septin": (2375, 345), "beta_1_6_glucan_biosynthesis": (2608, 345), "translation": (45, 520), "putative_mitochondria_protein": (278, 520), "mitochondrial_m_aaa_protease": (511, 520), "unassigned / uncertain": (744, 520)})
def stats_for(label):
    return unassigned_stats if label == "unassigned / uncertain" else count_lookup.get(label, (0, 0, 0, 0))
def fill_for(label):
    if label == "unassigned / uncertain": return "#f5d8cf"
    genes, responsive, _, _ = stats_for(label)
    ratio = responsive / genes if genes else 0
    return ["#f1f5f3", "#dcece5", "#b9dccd", "#83c2aa", "#42977b"][min(4, int(ratio * 10))]
def names_for(label):
    rows = machinery_final[machinery_final.subsystem.eq(label)]
    values = rows.gene_name.fillna(rows.yeast_ortholog).fillna(rows.ao_locus_tag).astype(str)
    return "; ".join(v.split(",")[0] for v in values)
# Draw the vector version first. Each pathway box includes gene and response counts.
svg = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>", "<defs><marker id='arrow' markerWidth='8' markerHeight='8' refX='7' refY='4' orient='auto'><path d='M0,0 L8,4 L0,8 z' fill='#61736c'/></marker></defs>", f"<rect width='{width}' height='{height}' fill='#fbfaf7'/><text x='45' y='42' font-family='Arial' font-size='26' font-weight='bold'>A. oryzae secretory machinery: primary pathway placement</text>", f"<text x='45' y='70' font-family='Arial' font-size='14' fill='#444'>{subtitle_1}</text><text x='45' y='90' font-family='Arial' font-size='14' fill='#444'>{subtitle_2.replace('<', '&lt;')}</text>"]
for left, right in zip(main_flow, main_flow[1:]):
    x1, y1 = positions[left]; x2, y2 = positions[right]
    svg.append(f"<line x1='{x1+box_w}' y1='{y1+box_h/2}' x2='{x2-7}' y2='{y2+box_h/2}' stroke='#61736c' stroke-width='3' marker-end='url(#arrow)'/>")
for branch in branches:
    x1, y1 = positions["golgi_processing"]; x2, y2 = positions[branch]
    svg.append(f"<path d='M {x1+box_w/2} {y1+box_h} V {y2-24} H {x2+box_w/2} V {y2-7}' fill='none' stroke='#61736c' stroke-width='2' marker-end='url(#arrow)'/>")
for label in labels:
    x, y = positions[label]
    key = None if label == "unassigned / uncertain" else label
    genes, responsive, up, down = stats_for(label)
    fill = fill_for(label)
    dash = " stroke-dasharray='7 5'" if key is None else ""
    title_lines = textwrap.wrap(label_map[label], width=25)
    tspans = ''.join(f"<tspan x='{x+12}' dy='{0 if n == 0 else 16}'>{line}</tspan>" for n, line in enumerate(title_lines))
    detail = f"{responsive} responsive: {up} up" + (f", {down} down" if down else "")
    names = names_for(label) if key is not None and genes <= 5 else ""
    name_lines = textwrap.wrap(names, width=27)[:2]
    name_svg = ''.join(f"<tspan x='{x+12}' dy='{14 if n else 0}'>{line}</tspan>" for n, line in enumerate(name_lines))
    svg.append(f"<g><title>{label_map[label]}</title><rect x='{x}' y='{y}' width='{box_w}' height='{box_h}' rx='8' fill='{fill}' stroke='#456' {dash}/><text x='{x+12}' y='{y+20}' font-family='Arial' font-size='13' font-weight='bold'>{tspans}</text><text x='{x+12}' y='{y+73}' font-family='Arial' font-size='13'>{genes} genes</text><text x='{x+12}' y='{y+94}' font-family='Arial' font-size='12' fill='#8a3b1c'>{detail}</text><text x='{x+12}' y='{y+112}' font-family='Arial' font-size='11' fill='#334'>{name_svg}</text></g>")
footer_1 = f"369 machinery genes; {369-unassigned_count} placed; {unassigned_count} unassigned; 51 responsive. Liu et al. 2014, doi:10.1186/1752-0509-8-73. Build {build_date}."
footer_2 = "Low-confidence placements are inferred rather than assigned by Liu; reviewed secondary roles are retained in the machinery table."
legend_x = 2050
svg.append(f"<text x='{legend_x}' y='535' font-family='Arial' font-size='12' font-weight='bold'>Responsive proportion</text>")
for i, (fill, text) in enumerate(zip(["#f1f5f3", "#dcece5", "#b9dccd", "#83c2aa", "#42977b"], ["0–9%", "10–19%", "20–29%", "30–39%", "40%+"])):
    svg.append(f"<rect x='{legend_x+i*145}' y='550' width='28' height='18' fill='{fill}' stroke='#456'/><text x='{legend_x+35+i*145}' y='564' font-family='Arial' font-size='11'>{text}</text>")
svg.append(f"<text x='45' y='700' font-family='Arial' font-size='12' fill='#555'>{footer_1}</text>")
svg.append(f"<text x='45' y='725' font-family='Arial' font-size='12' fill='#7a4b20'>{footer_2}</text></svg>")
Path("../data/processed/pathway_overview.svg").write_text("".join(svg), encoding="utf-8")
# Draw a PNG copy for browsers that do not preview SVG files reliably.
from PIL import Image, ImageDraw, ImageFont
canvas = Image.new("RGB", (width, height), "#fbfaf7")
draw = ImageDraw.Draw(canvas)
try:
    title_font, body_font, small_font = (ImageFont.truetype("arial.ttf", size) for size in (26, 15, 12))
except OSError:
    title_font = body_font = small_font = ImageFont.load_default()
draw.text((50, 24), "A. oryzae secretory machinery: primary pathway placement", fill="#223344", font=title_font)
draw.text((50, 62), subtitle_1, fill="#444444", font=small_font)
draw.text((50, 80), subtitle_2, fill="#444444", font=small_font)
for left, right in zip(main_flow, main_flow[1:]):
    x1, y1 = positions[left]; x2, y2 = positions[right]
    draw.line((x1+box_w, y1+box_h//2, x2-5, y2+box_h//2), fill="#61736c", width=3)
    draw.polygon([(x2-5, y2+box_h//2), (x2-13, y2+box_h//2-5), (x2-13, y2+box_h//2+5)], fill="#61736c")
for branch in branches:
    x1, y1 = positions["golgi_processing"]; x2, y2 = positions[branch]
    draw.line([(x1+box_w//2, y1+box_h), (x1+box_w//2, y2-24), (x2+box_w//2, y2-24), (x2+box_w//2, y2-5)], fill="#61736c", width=2)
    draw.polygon([(x2+box_w//2, y2-5), (x2+box_w//2-5, y2-13), (x2+box_w//2+5, y2-13)], fill="#61736c")
for label in labels:
    x, y = positions[label]
    key = None if label == "unassigned / uncertain" else label
    genes, responsive, up, down = stats_for(label)
    draw.rounded_rectangle((x, y, x + box_w, y + box_h), radius=8, fill=fill_for(label), outline="#445566", width=2)
    draw.multiline_text((x + 12, y + 9), "\n".join(textwrap.wrap(label_map[label], width=25)), fill="#223344", font=body_font, spacing=2)
    draw.text((x + 12, y + 70), f"{genes} genes", fill="#223344", font=small_font)
    detail = f"{responsive} responsive: {up} up" + (f", {down} down" if down else "")
    draw.text((x + 12, y + 90), detail, fill="#8a3b1c", font=small_font)
    if key is not None and genes <= 5: draw.multiline_text((x + 12, y + 106), "\n".join(textwrap.wrap(names_for(label), width=27)[:2]), fill="#334444", font=small_font, spacing=1)
draw.text((legend_x, 523), "Responsive proportion", fill="#223344", font=small_font)
for i, (fill, text) in enumerate(zip(["#f1f5f3", "#dcece5", "#b9dccd", "#83c2aa", "#42977b"], ["0-9%", "10-19%", "20-29%", "30-39%", "40%+"])):
    draw.rectangle((legend_x+i*145, 550, legend_x+28+i*145, 568), fill=fill, outline="#445566")
    draw.text((legend_x+35+i*145, 551), text, fill="#223344", font=small_font)
draw.text((45, 688), footer_1, fill="#555555", font=small_font)
draw.text((45, 713), footer_2, fill="#7a4b20", font=small_font)
canvas.save("../data/processed/pathway_overview.png")
assigned = int(machinery_final.subsystem.notna().sum())
ko_coverage = int(machinery_final.kegg_ko.notna().sum())
summary = f"""# Annotation coverage summary\n\nLiu supplied subsystem labels for 109 of 369 machinery genes. The 260 blanks still had yeast orthologs; most were additions from A. niger, A. oryzae, reciprocal-best-hit, or InParanoid source lists rather than rows in the original 16-subsystem yeast scaffold. They were therefore unlabeled, not lost.\n\nThis release places **{assigned}/369** genes. Assignments preserve their source in this order: Liu's explicit label, an unambiguous Liu description, the Feizi/Liu yeast scaffold, current UniProt annotation text, then KEGG pathway membership. Conflicts remain visible rather than being silently guessed.\n\nKEGG retrieval now validates the organism through `GET https://rest.kegg.jp/list/aor`. **{ko_coverage}/369** machinery genes have a supplied KO; unavailable values remain blank. KO family mappings add conservative glycosylation roles without replacing Liu-derived roles.\n\nFour reviewed secondary roles are retained directly in `secretion_machinery_genes.csv`. The pathway figure is built from that table and shows all {len(machinery_final) - assigned} unassigned genes.\n"""
Path("../data/processed/qa/coverage_summary.md").write_text(summary, encoding="utf-8")
print(f"Saved the pathway diagram and coverage summary. {assigned} of 369 genes have an assigned secretion stage.")

Saved the pathway diagram and coverage summary. 247 of 369 genes have an assigned secretion stage.
